# FakeBDTeen Fusion Trainer (Notebook 2)
This notebook loads the cached embeddings, trains the cross-attention model, and evaluates multi-label performance. Run after Notebook 1.

## 1. Dependencies & Feature Extraction
Import libraries, extract the feature ZIP, and load metadata.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score, hamming_loss
from tqdm import tqdm

dataset_root = '/kaggle/input/datasets/tanmoykdas/fakebdteen-extracted-feature-dataset/fakebdteen_extracted_features'
features_root = '/kaggle/working/'

os.makedirs(features_root, exist_ok=True)

def find_feature_root(search_root: str):
    for root, dirs, files in os.walk(search_root):
        if 'audio' in dirs and 'video' in dirs and 'fakebdteen_metadata.csv' in files:
            return root
    return None

features_zip = os.path.join(dataset_root, 'fakebdteen_extracted_features.zip')
if os.path.exists(features_zip):
    with zipfile.ZipFile(features_zip, 'r') as zf:
        zf.extractall('/kaggle/working')
    candidate_root = find_feature_root('/kaggle/working')
else:
    candidate_root = find_feature_root(dataset_root)

if candidate_root is None:
    raise FileNotFoundError(
        'Feature folders not found. Expected a folder containing audio/, video/, and fakebdteen_metadata.csv.'
    )

features_root = candidate_root
metadata_csv = os.path.join(features_root, 'fakebdteen_metadata.csv')

if not os.path.exists(metadata_csv):
    raise FileNotFoundError('fakebdteen_metadata.csv not found in features_root.')

registry_df = pd.read_csv(metadata_csv)
print(registry_df.head())
print(f'Total records: {len(registry_df)}')
print(f'Using features_root: {features_root}')

## 2. Speaker-Independent Split
Use GroupKFold to prevent speaker leakage between train and validation.

In [ ]:
gkf = GroupKFold(n_splits=5)
groups = registry_df['subject_id'].astype(str).values
splits = list(gkf.split(registry_df, registry_df[['video_label', 'audio_label']], groups))
train_idx, val_idx = splits[0]
train_df = registry_df.iloc[train_idx].reset_index(drop=True)
val_df = registry_df.iloc[val_idx].reset_index(drop=True)

train_speakers = sorted(train_df['subject_id'].unique().tolist())
val_speakers = sorted(val_df['subject_id'].unique().tolist())

print(f'Train speakers ({len(train_speakers)}): {train_speakers}')
print(f'Val speakers ({len(val_speakers)}): {val_speakers}')
print(f'Overlap: {set(train_speakers).intersection(set(val_speakers))}')

## 3. Dataset & DataLoaders
Load cached .npy features on the fly with a custom Dataset.

In [ ]:
class SavedFeaturesDataset(Dataset):
    def __init__(self, df: pd.DataFrame, features_root: str):
        self.df = df
        self.features_root = features_root
        self.audio_root = os.path.join(features_root, 'audio')
        self.video_root = os.path.join(features_root, 'video')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        unique_id = row['unique_id']
        audio_path = os.path.join(self.audio_root, f'{unique_id}.npy')
        video_path = os.path.join(self.video_root, f'{unique_id}.npy')
        audio_feat = np.load(audio_path).astype(np.float32)
        video_feat = np.load(video_path).astype(np.float32)
        audio_feat = torch.from_numpy(audio_feat)
        video_feat = torch.from_numpy(video_feat)
        video_label = torch.tensor(row['video_label'], dtype=torch.float32)
        audio_label = torch.tensor(row['audio_label'], dtype=torch.float32)
        return video_feat, audio_feat, video_label, audio_label

train_dataset = SavedFeaturesDataset(train_df, features_root)
val_dataset = SavedFeaturesDataset(val_df, features_root)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    pin_memory=True,
    num_workers=0
)

## 4. Cross-Attention Model
Define the fusion architecture and dual binary heads.

In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, embed_dim: int = 256, num_heads: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, q, k, v):
        attn_out, _ = self.attn(q, k, v, need_weights=False)
        return self.norm(q + attn_out)

class MultiModalDeepfakeDetector(nn.Module):
    def __init__(self, video_dim: int = 512, audio_dim: int = 1024, embed_dim: int = 256):
        super().__init__()
        self.video_proj = nn.Linear(video_dim, embed_dim)
        self.audio_proj = nn.Linear(audio_dim, embed_dim)
        self.cross_attn = CrossAttention(embed_dim=embed_dim, num_heads=4)
        self.video_head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
)
        self.audio_head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
)

    def forward(self, video_feat, audio_feat):
        video_emb = self.video_proj(video_feat)
        audio_emb = self.audio_proj(audio_feat)
        fused = self.cross_attn(video_emb, audio_emb, audio_emb)
        pooled = fused.mean(dim=1)
        video_logit = self.video_head(pooled).squeeze(-1)
        audio_logit = self.audio_head(pooled).squeeze(-1)
        return video_logit, audio_logit

## 5. Training
Train the multi-task model with AMP on GPU.

In [ ]:
import contextlib

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiModalDeepfakeDetector().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else torch.amp.GradScaler()
amp_ctx = torch.amp.autocast('cuda') if device.type == 'cuda' else contextlib.nullcontext()

epochs = 15
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for video_feat, audio_feat, video_label, audio_label in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}'):
        video_feat = video_feat.to(device, non_blocking=True)
        audio_feat = audio_feat.to(device, non_blocking=True)
        video_label = video_label.to(device, non_blocking=True)
        audio_label = audio_label.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with amp_ctx:
            video_logit, audio_logit = model(video_feat, audio_feat)
            loss_video = criterion(video_logit, video_label)
            loss_audio = criterion(audio_logit, audio_label)
            loss = loss_video + loss_audio
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * video_feat.size(0)
    avg_loss = running_loss / len(train_loader.dataset)
    print(f'Epoch {epoch} | Train Loss: {avg_loss:.6f}')

## 6. Multi-Label Evaluation
Compute multi-label metrics and per-head classification reports.

In [ ]:
model.eval()
all_video_logits = []
all_audio_logits = []
all_video_labels = []
all_audio_labels = []

with torch.no_grad():
    for video_feat, audio_feat, video_label, audio_label in tqdm(val_loader, desc='Validation'):
        video_feat = video_feat.to(device, non_blocking=True)
        audio_feat = audio_feat.to(device, non_blocking=True)
        video_logit, audio_logit = model(video_feat, audio_feat)
        all_video_logits.append(video_logit.cpu())
        all_audio_logits.append(audio_logit.cpu())
        all_video_labels.append(video_label.cpu())
        all_audio_labels.append(audio_label.cpu())

video_logits = torch.cat(all_video_logits).numpy()
audio_logits = torch.cat(all_audio_logits).numpy()
video_labels = torch.cat(all_video_labels).numpy()
audio_labels = torch.cat(all_audio_labels).numpy()

video_probs = 1 / (1 + np.exp(-video_logits))
audio_probs = 1 / (1 + np.exp(-audio_logits))

video_preds = (video_probs >= 0.5).astype(int)
audio_preds = (audio_probs >= 0.5).astype(int)

y_true = np.stack([video_labels, audio_labels], axis=1)
y_pred = np.stack([video_preds, audio_preds], axis=1)

subset_accuracy = accuracy_score(y_true, y_pred)
ham_loss = hamming_loss(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')

print(f'Subset Accuracy: {subset_accuracy:.4f}')
print(f'Hamming Loss: {ham_loss:.4f}')
print(f'Macro F1: {macro_f1:.4f}')

print('Video Head Classification Report')
print(classification_report(video_labels, video_preds, target_names=['Real', 'Fake']))
print('Audio Head Classification Report')
print(classification_report(audio_labels, audio_preds, target_names=['Real', 'Fake']))